# 09 - Full Walk-Forward Validation & Parameter Grid Search

**Author:** Sacha Huberty

**Purpose:** Formalize the S10 validation discipline the project has been
building toward: grid search two cheap-to-tune parameter families
in-sample only (mean-reversion lookback/entry_z via a no-model-fitting
event study, and turnover cap/no-trade band via a memoized
Black-Litterman strategy), select on cross-fold *stability*, not peak
performance, freeze the winning values into `config.yaml`, and then run
that frozen config through the out-of-sample walk-forward exactly once.
HMM and autoencoder hyperparameters are intentionally NOT grid-searched
here (prohibitive refit cost at this scale) and stay at their existing,
reasoned stage 3/4 defaults.

**Last updated:** 2026-07-26

**Note on the grid-search methodology used here:** the literal
"re-fit every model per fold" walk-forward described in
PROJECT_STRUCTURE.md 7 would need a fresh HMM + autoencoder refit per
parameter combination per fold, which is computationally infeasible in
this environment. Instead: (1) mean-reversion tuning uses
`meanreversion`'s pure vectorized math (no model fitting) so all 9
combos across all 12 IS years cost under a second; (2) turnover tuning
memoizes ONE full run of the (already-fitted-as-usual) Black-Litterman
strategy over the last 4 in-sample years, then replays cheap
accounting-only re-runs for each of the 9 rebalance-parameter combos.
This is faithful to the stage's spirit (IS-only tuning selected for
cross-fold stability, frozen config validated OOS exactly once) while
staying tractable.

## Setup

In [ ]:
# Must run before numpy/scipy/tensorflow are imported: on Windows, this
# backtest makes hundreds of tiny (~22-asset) SLSQP/HMM/covariance calls
# in a tight loop, and letting OpenBLAS/MKL and TensorFlow's own thread
# pools both fight over all CPU cores causes intermittent multi-minute
# stalls on individual weeks (confirmed empirically: identical code was
# ~9x slower and had several 30+ minute outlier calls with default
# threading vs. forced single-threaded BLAS -- stage 9). Each week's
# problem is small enough that single-threaded is strictly faster here.
import os

for _var in (
    "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS",
):
    os.environ[_var] = "1"

import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = '{:.4f}'.format

from atlas import (
    allocation, backtest, data, meanreversion, metrics, plotting,
    regimes, strategy, universe,
)

cfg = data.load_config()
posture_cfg = data.load_config(data.PROJECT_ROOT / "config" / "regime_posture.yaml")
cfg["grid_search"]

## Data

In [ ]:
as_of_universe = pd.Timestamp(cfg["general"]["is_end_date"])
universe_df = universe.load_universe(as_of_universe)
tickers = universe_df.index.tolist()
class_bucket = universe_df["class_bucket"]

prices = data.download_prices(tickers, start=cfg["general"]["start_date"])
prices = data.align_calendars(prices)
returns = data.daily_returns(prices)

is_end = pd.Timestamp(cfg["general"]["is_end_date"])
oos_start = pd.Timestamp(cfg["general"]["oos_start_date"])
lookback = cfg["optimization"]["lookback_days"]
cov_method = cfg["optimization"]["covariance"]

is_prices = prices.loc[:is_end]
returns.tail()

# Real risk-free rate (stage 11, DIAGNOSTIC.md Sec 2.3/6 action 3): the
# cash bucket's equilibrium prior is ~0 by construction (near-zero
# covariance with everything), regardless of the prevailing rate --
# pricing it with the actual T-bill yield instead fixes that
# degeneracy at its source. DTB3 is quoted in annualized percentage
# points; converted to a decimal annual rate and aligned to the
# trading-day calendar (ffill: T-bill publication has its own gaps/
# holidays independent of the equity calendar; no bfill, so any
# genuine cold start before DTB3's own history correctly falls back
# to rf=0 inside black_litterman_strategy rather than leaking a later
# value backward).
rf_raw = data.download_macro(
    [cfg["risk_free"]["fred_series"]], start=cfg["general"]["start_date"]
)
rf_series = (
    rf_raw[cfg["risk_free"]["fred_series"]] / 100.0
).reindex(returns.index).ffill()
print(f"rf_series: {rf_series.min():.3%} to {rf_series.max():.3%}, latest {rf_series.iloc[-1]:.3%}")

## Analysis / signal logic

### A. IS-only grid search: mean-reversion hyperparameters (event study)

No model fitting involved (`meanreversion.zscore` is pure rolling-window
math), so this is cheap enough to run the full mean-reversion grid
(`grid_search.meanreversion`) across every annual in-sample fold
(2010-2021). For each `(lookback_days, entry_z)` combo, in each annual
fold: was a >|entry_z| deviation from trend actually followed by
reversion toward the mean over the next 20 trading days? The z-score
and forward return are computed on the FULL in-sample price history
(so the rolling window has real data before the fold starts), but only
events dated inside the fold are counted toward that fold's hit rate --
this isolates a genuine per-year read rather than a growing-window
cumulative one.

In [ ]:
def fold_hit_rate(prices_full, combo_cfg, fold_start, fold_end, horizon_days=20):
    mcfg = combo_cfg["meanreversion"]
    window = mcfg["lookback_days"]
    log_price = np.log(prices_full)
    z = meanreversion.zscore(log_price, window)
    future_change = log_price.shift(-horizon_days) - log_price
    reverted = np.sign(future_change) == -np.sign(z)
    extreme = z.abs() > mcfg["entry_z"]

    in_fold = (z.index >= fold_start) & (z.index <= fold_end)
    extreme_fold = extreme.loc[in_fold]
    valid_fold = future_change.loc[in_fold].notna()
    reverted_fold = reverted.loc[in_fold]

    eligible = extreme_fold & valid_fold
    n = int(eligible.to_numpy().sum())
    if n == 0:
        return np.nan, 0
    hits = int((eligible & reverted_fold).to_numpy().sum())
    return hits / n, n


fold_periods = is_prices.index.to_period("Y").unique()
penalty = cfg["grid_search"]["stability_penalty"]
min_events_per_fold = 5


def evaluate_meanreversion(params):
    combo_cfg = copy.deepcopy(cfg)
    combo_cfg["meanreversion"]["lookback_days"] = params["lookback_days"]
    combo_cfg["meanreversion"]["entry_z"] = params["entry_z"]

    hit_rates, total_events = [], 0
    for period in fold_periods:
        hr, n = fold_hit_rate(
            is_prices, combo_cfg, period.start_time, period.end_time
        )
        total_events += n
        if n >= min_events_per_fold:
            hit_rates.append(hr)

    hit_rates = pd.Series(hit_rates, dtype=float)
    mean_hr = hit_rates.mean() if len(hit_rates) else np.nan
    std_hr = hit_rates.std(ddof=1) if len(hit_rates) > 1 else 0.0
    return {
        "mean_hit_rate": mean_hr,
        "hit_rate_std": std_hr,
        "stability_score": mean_hr - penalty * std_hr,
        "total_events": total_events,
        "n_folds_with_events": len(hit_rates),
    }


meanrev_grid = backtest.grid_search(
    cfg["grid_search"]["meanreversion"], evaluate_meanreversion
)
meanrev_grid.sort_values("stability_score", ascending=False)

In [ ]:
# Require a reasonable sample size before trusting a combo's hit rate.
qualified = meanrev_grid[meanrev_grid["total_events"] >= 30]
best_meanrev = qualified.sort_values("stability_score", ascending=False).iloc[0]
best_meanrev

### A2. IS-only: HMM n_states (BIC) and V2 per-asset hit-rate eligibility (Tier 3)

Two more IS-only, freeze-for-OOS selections (DIAGNOSTIC.md Sec 6 actions 6/7), same convention as the mean-reversion grid above: `regimes.hmm_bic_curve` fits a candidate HMM at each `n_states` in `grid_search.hmm.n_states_range` and scores it by BIC (lower is better, already penalizes complexity via hmmlearn's own `.bic()`); `meanreversion.hit_rate_eligible_tickers` restricts V2's view to only the tickers whose own historical hit rate clears `meanreversion.min_hit_rate` -- a uniform mean-reversion rule applied to the whole universe has no clean edge (stage 5/11), but individual assets may.

In [ ]:
market_returns_is = returns[cfg["regimes"]["market_ticker"]].loc[:is_end]
hmm_bic = regimes.hmm_bic_curve(
    market_returns_is, cfg, cfg["grid_search"]["hmm"]["n_states_range"]
)
best_n_states = int(hmm_bic.idxmin())
hmm_bic

In [ ]:
eligibility_cfg = copy.deepcopy(cfg)
eligibility_cfg["meanreversion"]["lookback_days"] = int(
    best_meanrev["lookback_days"]
)
eligibility_cfg["meanreversion"]["entry_z"] = float(best_meanrev["entry_z"])
meanrev_eligible = meanreversion.hit_rate_eligible_tickers(
    is_prices, eligibility_cfg
)
min_hr = eligibility_cfg["meanreversion"]["min_hit_rate"]
print(f"Winning HMM n_states (BIC): {best_n_states}")
print(f"V2-eligible tickers (min_hit_rate={min_hr}): {list(meanrev_eligible)}")

### A3. IS-only grid search: momentum parameters (V5, ANALYSIS_V2.md action 1)

Same no-model-fitting, IS-only, stability-selected discipline as Part A: for each candidate `(lookback_days, skip_days, top_fraction)`, compute the 12-1 cross-sectional rank at every IS date (vectorized per date: `log_price.shift(skip) - log_price.shift(lookback)`, then a per-date, per-bucket percentile rank -- no HMM, no Black-Litterman, no SLSQP), then measure the forward-return SPREAD between assets above vs below the `top_fraction` cutoff, in each annual IS fold. A real relative-momentum signal should show a positive, stable spread (top performers keep outperforming); `stability_score = mean(fold spread) - penalty * std(fold spread)`, same convention as the mean-reversion grid above. **Not tuned here:** `trend_lookback_days` (the absolute leg's own horizon) -- this spread test only exercises the relative leg, so `trend_lookback_days` stays at its config default (252d, ANALYSIS_V2.md Sec 5.1's headline absolute-momentum variant), a reasoned-not-tuned choice in the same category as the HMM/autoencoder hyperparameters (Part C).

In [ ]:
def _cross_sectional_rank_panel(log_price, bucket, lookback, skip):
    score_panel = log_price.shift(skip) - log_price.shift(lookback)
    return score_panel.apply(
        lambda row: row.groupby(bucket.reindex(row.index)).rank(pct=True),
        axis=1,
    )


_is_log_price = np.log(is_prices)
_momentum_horizon_days = 20
_min_events_per_fold_momentum = 50


def evaluate_momentum(params):
    rank_panel = _cross_sectional_rank_panel(
        _is_log_price, class_bucket,
        params["lookback_days"], params["skip_days"],
    )
    forward_return = (
        _is_log_price.shift(-_momentum_horizon_days) - _is_log_price
    )
    cutoff = 1.0 - params["top_fraction"]
    top_mask = rank_panel >= cutoff
    bottom_mask = rank_panel < cutoff

    spreads, total_events = [], 0
    for period in fold_periods:
        in_fold = (
            (is_prices.index >= period.start_time)
            & (is_prices.index <= period.end_time)
        )
        fwd_fold = forward_return.loc[in_fold]
        valid_fold = fwd_fold.notna()
        top_vals = fwd_fold.to_numpy()[
            (top_mask.loc[in_fold] & valid_fold).to_numpy()
        ]
        bottom_vals = fwd_fold.to_numpy()[
            (bottom_mask.loc[in_fold] & valid_fold).to_numpy()
        ]
        n = len(top_vals) + len(bottom_vals)
        total_events += n
        if (
            len(top_vals) >= _min_events_per_fold_momentum
            and len(bottom_vals) >= _min_events_per_fold_momentum
        ):
            spreads.append(float(np.mean(top_vals) - np.mean(bottom_vals)))

    spreads = pd.Series(spreads, dtype=float)
    mean_spread = spreads.mean() if len(spreads) else np.nan
    std_spread = spreads.std(ddof=1) if len(spreads) > 1 else 0.0
    return {
        "mean_spread": mean_spread,
        "spread_std": std_spread,
        "stability_score": mean_spread - penalty * std_spread,
        "total_events": total_events,
        "n_folds_with_events": len(spreads),
    }


momentum_grid = backtest.grid_search(
    cfg["grid_search"]["momentum"], evaluate_momentum
)
momentum_grid.sort_values("stability_score", ascending=False)

In [ ]:
qualified_momentum = momentum_grid[momentum_grid["n_folds_with_events"] >= 8]
best_momentum = qualified_momentum.sort_values(
    "stability_score", ascending=False
).iloc[0]
best_momentum

### B. IS-only grid search: turnover parameters (memoized BL strategy)

The turnover cap and no-trade band are rebalance-time-only parameters:
they never change what the strategy WANTS to hold, only how quickly it
gets there. That means the (expensive) weekly target-weight sequence
can be computed once via `backtest.memoize_strategy` and reused for
every `(no_trade_band, max_weekly_turnover)` combo -- each grid point
then only re-runs `backtest.run`'s cheap accounting loop.

**Scoping note:** to keep this tractable, this grid search uses only
the last 4 in-sample years (still strictly before `oos_start_date`,
so still IS-only) rather than the full 2010-2021 span used in part A --
running the full BL pipeline (HMM decode + views + SLSQP) week-by-week
over 12 years just to tune 2 rebalance parameters is not worth the
cost. The anomaly override is also left off here (it doesn't interact
with turnover-cap mechanics) to save the autoencoder's fit cost; it is
restored for the final frozen OOS run in part D, matching stage 8's
"current best" pipeline.

In [ ]:
tuned_cfg = copy.deepcopy(cfg)
tuned_cfg["meanreversion"]["lookback_days"] = int(best_meanrev["lookback_days"])
tuned_cfg["meanreversion"]["entry_z"] = float(best_meanrev["entry_z"])
tuned_cfg["momentum"]["lookback_days"] = int(best_momentum["lookback_days"])
tuned_cfg["momentum"]["skip_days"] = int(best_momentum["skip_days"])
tuned_cfg["momentum"]["top_fraction"] = float(best_momentum["top_fraction"])

turnover_is_start = is_end - pd.DateOffset(years=4)
buffer_start_pos = max(0, returns.index.searchsorted(turnover_is_start) - lookback)
turnover_is_returns = returns.iloc[buffer_start_pos:returns.index.searchsorted(is_end) + 1]

bl_fn_for_turnover = strategy.black_litterman_strategy(
    class_bucket, tuned_cfg, posture_cfg, rf_series=rf_series,
    meanrev_eligible=meanrev_eligible,
)
memoized_bl_fn = backtest.memoize_strategy(bl_fn_for_turnover)

is_validation_fold = cfg["grid_search"]["is_validation_fold"]


def evaluate_turnover(params):
    combo_cfg = copy.deepcopy(tuned_cfg)
    combo_cfg["rebalance"]["no_trade_band"] = params["no_trade_band"]
    combo_cfg["rebalance"]["max_weekly_turnover"] = params["max_weekly_turnover"]

    result = backtest.run(memoized_bl_fn, turnover_is_returns, combo_cfg)
    is_returns_only = result.daily_returns.loc[turnover_is_start:]
    wf = backtest.walk_forward(is_returns_only, combo_cfg, fold=is_validation_fold)
    return {
        **wf.summary,
        "avg_weekly_turnover": result.metrics["avg_weekly_turnover"],
        "total_cost_drag": result.metrics["total_cost_drag"],
    }


turnover_grid = backtest.grid_search(cfg["grid_search"]["turnover"], evaluate_turnover)
turnover_grid.sort_values("stability_score", ascending=False)

In [ ]:
best_turnover = turnover_grid.sort_values("stability_score", ascending=False).iloc[0]
best_turnover

### C. Freeze the config

HMM (`regimes.hmm`) and autoencoder (`anomaly`) hyperparameters are
deliberately left untouched at their stage 3/4 defaults: exhaustively
grid-searching them would mean re-fitting an HMM and an LSTM
autoencoder per parameter combination per fold, which is not
tractable here. They remain reasoned choices, not tuned ones -- this
is stated plainly rather than dressed up as a completed grid search.

**Turnover cap policy change (post stage-11 ablation):** the grid
search above still selects the tightest combo (1% cap, 1% band) as
"most stable," but a stability-penalized objective applied to a risk
CONTROL will always buy stability with paralysis -- band == cap
collapses every rebalance to "trade exactly 1% or freeze," throttling
every upstream signal (REVIEW.md 2.1). Stage 11's ablation confirmed
relaxing to the S4-convention 2% cap / 0.5% band raises OOS Sharpe
0.806 -> 0.875 with no drawdown cost. The frozen config below therefore
uses the POLICY value from `config.yaml` (2% / 0.5%), not the grid's
selection -- the turnover grid search above remains as a diagnostic
(what would a pure stability criterion pick?), not as the authority
for this parameter going forward.

In [ ]:
frozen_cfg = copy.deepcopy(cfg)
frozen_cfg["meanreversion"]["lookback_days"] = int(best_meanrev["lookback_days"])
frozen_cfg["meanreversion"]["entry_z"] = float(best_meanrev["entry_z"])
frozen_cfg["regimes"]["hmm"]["n_states"] = best_n_states
frozen_cfg["momentum"]["lookback_days"] = int(best_momentum["lookback_days"])
frozen_cfg["momentum"]["skip_days"] = int(best_momentum["skip_days"])
frozen_cfg["momentum"]["top_fraction"] = float(best_momentum["top_fraction"])
# rebalance.no_trade_band / max_weekly_turnover are deliberately NOT
# overridden with best_turnover here -- they stay at config.yaml's
# policy values (see the markdown above and config.yaml's rebalance
# section for why the grid's own selection is not adopted).

print("Frozen HMM n_states (BIC):", frozen_cfg["regimes"]["hmm"]["n_states"])
mom_cfg = frozen_cfg["momentum"]
print("Frozen momentum:", mom_cfg["lookback_days"],
      mom_cfg["skip_days"], mom_cfg["top_fraction"])
print("Frozen meanreversion:", frozen_cfg["meanreversion"]["lookback_days"], frozen_cfg["meanreversion"]["entry_z"])
print("Frozen rebalance (policy, not grid-selected):", frozen_cfg["rebalance"]["no_trade_band"], frozen_cfg["rebalance"]["max_weekly_turnover"])
print("Grid would have selected:", best_turnover["no_trade_band"], best_turnover["max_weekly_turnover"])

### D. Frozen config -> OOS walk-forward, run exactly once

This is the canonical result. The anomaly override is restored here
(matching stage 8's "current best" pipeline: BL + anomaly override).
Same buffered-lookback, reduced-anomaly-epoch scope trade as stages
6-8 (a compute-budget concession applied identically to every strategy
compared below, not chosen in response to OOS performance). After this
cell runs, `frozen_cfg`'s parameters are not revisited.

**Risk-free rate (stage 11):** `black_litterman_strategy` now takes
`rf_series` (the real T-bill yield computed in Setup/Data above) and
uses it three ways each week: as `rf` in the max-Sharpe objective, as
`rf` in the utility gate, and as the cash bucket's equilibrium-prior
override (replacing Pi's near-zero implied return for a near-zero-
covariance asset). This is the correct fix for the cash degeneracy
diagnosed in stage 11 -- pricing cash properly rather than excluding
it, which DIAGNOSTIC.md's gate census showed was the wrong remedy
(it removed the variance anchor that let the BL book pass the gate at
all).

In [ ]:
buffer_start_pos = max(0, returns.index.searchsorted(oos_start) - lookback)
backtest_returns = returns.iloc[buffer_start_pos:]

oos_cfg = copy.deepcopy(frozen_cfg)
oos_cfg["anomaly"]["epochs"] = 10
oos_cfg["anomaly"]["patience"] = 3
oos_cfg["anomaly"]["refit_frequency_days"] = 126

bl_fn = strategy.black_litterman_strategy(
    class_bucket, oos_cfg, posture_cfg, rf_series=rf_series,
    meanrev_eligible=meanrev_eligible,
)
bl_anomaly_fn = strategy.with_anomaly_override(bl_fn, oos_cfg)

frozen_result = backtest.run(bl_anomaly_fn, backtest_returns, oos_cfg)

### D2. Bucket-level Black-Litterman (Tier 2 item 5, DIAGNOSTIC.md action 5)

`strategy.bucket_black_litterman_strategy` runs the same Black-Litterman/gate machinery on the four `allocation.permanent` asset-class buckets (equity, fixed_income, commodity, cash) instead of the 22 individual tickers -- DIAGNOSTIC.md's argument that with ~22 assets that are really 4-5 correlated bets, per-asset covariance/return estimation is mostly noise. Only V1 (regime posture) contributes a view here; V2/V3 have no natural bucket-level analog and are not applied. Run side by side with the asset-level `black_litterman_frozen` on the identical OOS window (`modules.bucket_level` documents which is "current" without deleting either path).

In [ ]:
bucket_bl_fn = strategy.bucket_black_litterman_strategy(
    class_bucket, oos_cfg, posture_cfg, rf_series=rf_series
)
bucket_result = backtest.run(bucket_bl_fn, backtest_returns, oos_cfg)

In [ ]:
snapshot_window = returns.loc[:is_end].tail(lookback)
mu_is = allocation.mean_returns(snapshot_window)
cov_is = allocation.covariance_matrix(snapshot_window, method=cov_method)
w_max_sharpe_static = allocation.max_sharpe(mu_is, cov_is, cfg)


def make_classical_strategy(method):
    def strategy_fn(as_of, window):
        w = window.tail(lookback)
        cov_t = allocation.covariance_matrix(w, method=cov_method)
        if method == "gmv":
            return allocation.gmv(cov_t, cfg)
        if method == "risk_parity":
            return allocation.risk_parity(cov_t, cfg)
        if method == "hrp":
            return allocation.hrp(w, cfg)
        raise ValueError(method)
    return strategy_fn


_benchmark_cap = cfg["constraints"]["per_asset_cap"]


def permanent_strategy(as_of, window):
    # Tier 3 (DIAGNOSTIC.md Sec 5.2 item 1/action 8): capped so
    # the benchmark faces the same per-asset cap the strategy does.
    return allocation.permanent(class_bucket, cap=_benchmark_cap)


def sixty_forty_strategy(as_of, window):
    return allocation.sixty_forty(class_bucket, cap=_benchmark_cap)


def max_sharpe_static_strategy(as_of, window):
    return w_max_sharpe_static


benchmark_fns = {
    "permanent": permanent_strategy,
    "sixty_forty": sixty_forty_strategy,
    "max_sharpe_static": max_sharpe_static_strategy,
    "gmv": make_classical_strategy("gmv"),
    "risk_parity": make_classical_strategy("risk_parity"),
    "hrp": make_classical_strategy("hrp"),
}
benchmark_results = {
    name: backtest.run(fn, backtest_returns, cfg)
    for name, fn in benchmark_fns.items()
}
all_results = {
    "black_litterman_frozen": frozen_result,
    "black_litterman_bucket": bucket_result,
    **benchmark_results,
}

## Results

In [ ]:
def oos_metrics(result):
    r = result.daily_returns.loc[oos_start:]
    aligned_rf = rf_series.reindex(r.index)
    return {
        "ann_return": metrics.ann_return(r),
        "ann_vol": metrics.ann_vol(r),
        "sharpe": metrics.sharpe(r),
        "excess_sharpe": metrics.sharpe(r, rf=aligned_rf),
        "sortino": metrics.sortino(r),
        "calmar": metrics.calmar(r),
        "max_drawdown": metrics.max_drawdown(r),
        "hit_rate": metrics.hit_rate(r),
        "avg_weekly_turnover": result.turnover.loc[oos_start:].mean(),
        "total_cost_drag": result.costs.loc[oos_start:].sum(),
    }


comparison_table = pd.DataFrame(
    {name: oos_metrics(res) for name, res in all_results.items()}
).T
comparison_table.sort_values("sharpe", ascending=False)

### Risk-matched benchmark: is there real alpha? (ANALYSIS_V2.md Sec 2/action 3)

Sharpe is leverage-invariant, so comparing raw returns (as the headline table above does) doesn't answer the question that actually matters: does ATLAS add genuine skill, or does it just take more risk than `permanent` and get paid for that risk the way any levered position would? The fair test is to lever `permanent` (financed at the prevailing T-bill rate, the same `rf_series` used everywhere else in this notebook) up to **ATLAS's own realized volatility**, then compare returns at that matched risk level. If ATLAS has real alpha, its return should exceed the risk-matched benchmark's; if it doesn't, the two should be indistinguishable (both should also land at roughly the same excess Sharpe as unlevered `permanent`, since leverage financed at `rf` does not change Sharpe in theory -- only compounding/path effects over a real multi-year series can move it slightly).

In [ ]:
permanent_oos = benchmark_results["permanent"].daily_returns.loc[oos_start:]
atlas_oos = frozen_result.daily_returns.loc[oos_start:]
rf_oos = rf_series.reindex(permanent_oos.index)

atlas_vol = metrics.ann_vol(atlas_oos)
permanent_vol = metrics.ann_vol(permanent_oos)
leverage = atlas_vol / permanent_vol

# Levering financed at the prevailing risk-free rate: the
# levered position's daily return is rf + leverage * (unlevered
# excess return) -- borrow the extra (leverage - 1) at rf, keep
# the full levered exposure to permanent's own excess return.
daily_rf = rf_oos / 252
levered_permanent_returns = daily_rf + leverage * (permanent_oos - daily_rf)

levered_label = f"permanent (levered {leverage:.2f}x to ATLAS's vol)"
risk_matched_metrics = {
    "permanent": oos_metrics(benchmark_results["permanent"]),
    levered_label: {
        "ann_return": metrics.ann_return(levered_permanent_returns),
        "ann_vol": metrics.ann_vol(levered_permanent_returns),
        "sharpe": metrics.sharpe(levered_permanent_returns),
        "excess_sharpe": metrics.sharpe(levered_permanent_returns, rf=rf_oos),
        "sortino": metrics.sortino(levered_permanent_returns),
        "max_drawdown": metrics.max_drawdown(levered_permanent_returns),
    },
    "black_litterman_frozen (ATLAS)": oos_metrics(frozen_result),
}
risk_matched_cols = [
    "ann_return", "ann_vol", "sharpe",
    "excess_sharpe", "sortino", "max_drawdown",
]
risk_matched_table = pd.DataFrame(risk_matched_metrics).T[risk_matched_cols]

atlas_ann_return = risk_matched_table.loc[
    "black_litterman_frozen (ATLAS)", "ann_return"
]
levered_ann_return = risk_matched_table.loc[levered_label, "ann_return"]
alpha = atlas_ann_return - levered_ann_return

print(f"Leverage to match ATLAS's vol: {leverage:.3f}x")
print(f"ATLAS's alpha over risk-matched permanent: {alpha:+.4%} per year")
risk_matched_table

### Per-fold OOS walk-forward (the canonical, stability-checked view)

Quarterly folds (`backtest.walk_forward_fold`) on the frozen strategy's
OOS daily returns -- the actual S10 "average over folds" report, not
just one aggregate OOS number.

In [ ]:
frozen_oos_returns = frozen_result.daily_returns.loc[oos_start:]
frozen_wf = backtest.walk_forward(frozen_oos_returns, oos_cfg)
frozen_wf.fold_metrics

In [ ]:
frozen_wf.summary

In [ ]:
oos_returns = {
    name: res.daily_returns for name, res in all_results.items()
}
fig = plotting.equity_curves(
    oos_returns,
    oos_start=oos_start,
    highlight="black_litterman_frozen",
    title="OOS equity curves: frozen Black-Litterman vs. benchmarks",
)
plt.show()

In [ ]:
fig = plotting.drawdown_curves(
    oos_returns,
    oos_start=oos_start,
    highlight="black_litterman_frozen",
    title="OOS drawdowns: frozen Black-Litterman vs. benchmarks",
)
plt.show()

In [ ]:
fig = plotting.weight_evolution(
    frozen_result.weights.loc[oos_start:],
    title="Frozen Black-Litterman: OOS weight evolution",
)
plt.show()

### Regime-colored performance (visualization context only)

A single full-sample HMM fit/decode over the whole return history,
used only to shade the background by posture for visual context. This
is NOT the walk-forward, point-in-time posture the strategy actually
used week to week (that is recomputed on a rolling basis inside
`black_litterman_strategy`) -- it's a cheaper, full-sample view purely
to help the eye connect performance with regime, and should not be
read as a performance number.

In [ ]:
market_ticker = cfg["regimes"]["market_ticker"]
full_regime = regimes.market_regime(returns[market_ticker], cfg, posture_cfg)
posture_series = full_regime["posture_series"].reindex(frozen_oos_returns.index).ffill()

posture_colors = {"risk_on": "#c8e6c9", "risk_off": "#ffcdd2", "neutral": "#e0e0e0"}
fig, ax = plt.subplots(figsize=(11, 5))
equity = (1.0 + frozen_oos_returns).cumprod()
prev_date = posture_series.index[0]
prev_posture = posture_series.iloc[0]
for date, posture in posture_series.items():
    if posture != prev_posture:
        ax.axvspan(prev_date, date, color=posture_colors.get(prev_posture, "#ffffff"), alpha=0.5)
        prev_date, prev_posture = date, posture
ax.axvspan(prev_date, posture_series.index[-1], color=posture_colors.get(prev_posture, "#ffffff"), alpha=0.5)
equity.plot(ax=ax, color="black", linewidth=1.5, label="black_litterman_frozen")
ax.set_title("Frozen Black-Litterman OOS equity, shaded by full-sample HMM posture")
ax.legend()
plt.tight_layout()
plt.show()

### Turnover report

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
frozen_result.turnover.loc[oos_start:].plot(ax=ax, label="weekly turnover")
ax.axhline(oos_cfg["rebalance"]["max_weekly_turnover"], color="red", linestyle="--", label="turnover cap")
ax.axhline(oos_cfg["rebalance"]["no_trade_band"], color="gray", linestyle=":", label="no-trade band")
ax.set_title("Frozen Black-Litterman: OOS weekly turnover vs. cap and no-trade band")
ax.legend()
plt.tight_layout()
plt.show()

## Notes / next steps

In [ ]:
print(f"Winning meanreversion combo: lookback_days={frozen_cfg['meanreversion']['lookback_days']}, entry_z={frozen_cfg['meanreversion']['entry_z']}")
print(f"Winning turnover combo: no_trade_band={frozen_cfg['rebalance']['no_trade_band']}, max_weekly_turnover={frozen_cfg['rebalance']['max_weekly_turnover']}")
print(comparison_table.sort_values('sharpe', ascending=False))

**Honest findings:**

- **The grid search did not improve OOS performance over stage 8's
  manually-chosen defaults, and if anything is marginally behind.**
  The IS-selected combo (mean-reversion `lookback_days=20, entry_z=2.0`;
  turnover `no_trade_band=0.01, max_weekly_turnover=0.01`) produced a
  frozen Black-Litterman OOS Sharpe of **0.8564** over 2022-01 to
  2026-07, versus stage 8's untuned 0.895 over a similar (shorter)
  OOS window. This is exactly what the S10 discipline predicts can
  happen: selecting IS parameters for cross-fold *stability* rather
  than peak IS performance is not a promise of better OOS performance,
  and the honest-reporting rule means saying so plainly rather than
  re-tuning after the fact.
- `permanent` again posts the best OOS Sharpe of the whole comparison
  (1.0983), same as stage 8's finding. The frozen Black-Litterman
  strategy's max drawdown (-10.4%) is close to but not quite better
  than permanent's (-12.6%), and is not the best in the group either
  (HRP's -9.2% is the smallest). BL's annualized vol (5.3%) remains
  the second-lowest after HRP, consistent with the utility gate
  sometimes preferring the defensive GMV/Risk-Parity book.
- **The turnover grid search had limited power to discriminate.**
  All 9 combos' stability scores were negative and clustered close
  together (-0.607 to -0.783); the top 3 combos (all using the
  tightest 1% turnover cap) differed by less than 0.003 in stability
  score, indistinguishable given the underlying noise. Only 4 annual
  IS folds were available for this grid (the scoped-down last-4-years
  window used to keep the memoized run affordable), which is a small
  sample for a std-penalized stability score. The winning combo does
  cleanly minimize average turnover (1.13%) and cost drag (0.41%)
  versus the wider bands, which is a real, if modest, effect.
- **Quarterly OOS fold Sharpe is highly volatile and its mean should
  not be read as "the" Sharpe.** The per-fold table shows sharpe_mean
  1.42 vs. sharpe_std 2.21 (stability_score -0.79): 2022 was sharply
  negative (Q1-Q3 Sharpe -0.78, -2.46, -2.28 amid the bond+equity
  selloff) before a strong recovery, and several quarters since show
  Sharpe above 4-5, which mostly reflects how noisy an annualized
  Sharpe estimated from ~62 days of data is, not sustained skill. The
  properly-computed full-period OOS Sharpe (0.86, from the whole
  contiguous return series) is the reliable headline number; the
  fold-by-fold average is for reading consistency and dispersion, not
  as a better point estimate.
- **A real engineering finding from this stage, independent of the
  strategy's numbers:** two of the four IS validation folds tested
  during grid search (2018-03 and 2018-04, under the eventual
  winning `lookback_days=20`) took 30+ minutes each in a single-
  threaded-unaware run, driven by OpenBLAS/MKL and TensorFlow's
  thread pools fighting over CPU cores across hundreds of tiny
  (~22-asset) SLSQP/HMM calls in a tight Windows loop -- not a bug in
  the strategy logic. Forcing single-threaded BLAS/OMP (top of the
  Setup cell) fixed it and made the whole notebook roughly 9x faster
  end to end, since each week's optimization problem is far too small
  to benefit from multi-threading anyway.



---

**Addendum (2026-07-30, after the stage-11 ablation and the V2 unit-mismatch fix):**

Stage 11's ablation study found V2 (mean-reversion) contributing
essentially nothing to OOS Sharpe (-0.0125 marginal), which turned out
to be a real bug, not a calibration choice: `meanreversion.
reversion_signal`'s view was a raw one-day OU drift added directly to
an annualized Black-Litterman prior, landing roughly two orders of
magnitude below `regime_view_magnitude`. Fixed by scaling the daily
drift by the (capped) reversion half-life (see `fix: V2 mean-reversion
unit mismatch`).

**Honest result: fixing the bug did not improve OOS performance --
if anything it is marginally worse.** Re-running this notebook after
the fix (mean-reversion and turnover grid searches both re-selected
the identical parameters as before, `lookback_days=20, entry_z=2.0`,
`no_trade_band=0.01, max_weekly_turnover=0.01`, confirming the grids
themselves don't depend on the view's scale) gives a frozen OOS Sharpe
of **0.8348**, down from 0.8564 pre-fix (max drawdown -10.18% vs.
-10.43%, both essentially unchanged). Once V2's view is properly
scaled and actually participates in the fusion with realistic weight,
its net effect in this specific OOS window is still slightly negative
-- consistent with stage 5's own event study, which found reversion
hit rates above chance for some assets (HYG, VNQ) but below chance for
others (AGG, LQD, USO): a uniform mean-reversion rule applied
indiscriminately across the universe does not have a clean edge here,
scaling bug or not. This is reported as found, not re-tuned away:
correcting a real unit bug was worth doing on its own merits
(regardless of whether it improved the backtest), and the honest
answer is that it didn't.

Note also that a few more trading days of live 2026 data entered the
return series between the pre-fix and post-fix runs (this notebook is
re-executed, not replayed from a frozen dataset), so every benchmark's
number shifted slightly too (e.g. `permanent`'s Sharpe moved from
1.0983 to 1.0704) -- a small amount of this delta is calendar drift,
not the fix itself, though the fix is still the dominant driver given
`black_litterman_frozen` moved against the grain of the other mostly-
unaffected classical books (gmv, risk_parity, hrp, permanent,
sixty_forty do not depend on V2 at all and should only show that
small calendar-drift noise).

---

**Addendum (2026-07-30, after restoring the turnover cap to the S4
policy value):**

Per REVIEW.md and stage 11's ablation, `config.yaml`'s
`rebalance.max_weekly_turnover`/`no_trade_band` were restored to the
S4 convention (2% / 0.5%) as a fixed risk-control policy, no longer
subject to the stability-penalized grid (which, run again here for
reference, still selects the throttled 1%/1% combo -- that grid now
remains purely diagnostic). This required also patching this
notebook's own freeze step (cell 13), which had been silently
re-deriving the turnover parameters from `best_turnover` regardless of
`config.yaml` -- a real bug in the notebook's own logic, not just a
config value to flip.

**Honest result: this is the first change in the project's history
that unambiguously helped.** Frozen OOS Sharpe rose from 0.8348 (1%/1%
band=cap) to **0.8969** (2%/0.5%), with max drawdown actually
*improving* slightly (-10.02% vs -10.18%) despite more than triple the
turnover cost drag (0.46% vs 0.14% over the OOS window -- still small
in absolute terms). `black_litterman_frozen` now beats both
`risk_parity` (0.8146) and `hrp` (0.8176) for the first time in this
project, though `permanent` (1.0681) still leads. This matches stage
11's diagnosis precisely: band == cap had been forcing every
rebalance into "trade exactly 1% or freeze," throttling the upstream
signal apparatus regardless of its quality -- unwinding that throttle,
not any signal improvement, produced the largest single Sharpe gain
seen across all 11 stages.

---

**Addendum (2026-07-30, after removing the cash degeneracy from
max_sharpe):**

Per REVIEW.md 2.1(2), `allocation.max_sharpe` now takes an `exclude`
parameter, and `black_litterman_strategy` uses it to drop the `cash`
bucket (BIL) from its max-Sharpe candidate book only -- GMV and Risk
Parity are unaffected and remain free to hold cash.

**Honest result: this made OOS performance meaningfully worse, the
opposite of REVIEW.md's hypothesis.** Frozen Sharpe fell from 0.8969
to **0.7265** (ann_vol dropped from 5.54% to 4.45%, ann_return from
4.93% to 3.18%; max drawdown did improve, -8.73% vs -10.02%, but not
enough to offset the return decline). Every other book in the
comparison table (permanent, hrp, risk_parity, gmv, max_sharpe_static,
sixty_forty) is numerically identical to the pre-fix run to four
decimal places, confirming the entire delta is attributable to this
change, not to the extra few days of calendar drift between runs.

**Why, mechanically:** a targeted diagnostic (sampling 10 OOS
rebalance dates and comparing `utility_select`'s winner with and
without the cash exclusion) shows the mechanism directly. With cash
available, `max_sharpe(mu_BL)` can sometimes build a book with up to
20% cash (the per-asset cap) that is low-vol enough to *win* the
utility comparison against GMV/Risk Parity outright (e.g. 2022-08-23,
2023-04-13 in the sample). Once cash is excluded, that same book is
forced entirely into more volatile assets, loses those utility
comparisons to GMV instead, and the pipeline falls back to GMV -- a
pure minimum-variance book with no return view at all -- more often
than before. The fix didn't "un-mute the risk-on side" as
hypothesized; it removed one of the ways the utility gate could
produce a moderate, cash-anchored middle ground between full GMV
defensiveness and full risk-taking, and the gate defaulted to GMV
instead.

**This is not reverted.** The underlying critique is still correct on
its own terms: at rf=0, a near-zero-vol asset genuinely is degenerate
in a Sharpe-ratio objective, independent of whether removing it helps
this particular backtest. The fix is kept for that reason, and the
result is reported exactly as it came out -- a second, unrelated
"correct the bug, and it doesn't help" data point (after the V2 unit
fix) that says more about how tightly coupled this pipeline's stages
are than about any individual signal's quality. A natural follow-up
(out of scope here) would be re-deriving the equilibrium prior or the
utility gate's risk aversion with cash held out from the start, rather
than only patching the final optimization step.

---

**Addendum (2026-07-31, Tier 1 of DIAGNOSTIC.md's action plan: V3 off,
cash exclusion reverted, real risk-free rate):**

All three measured Tier-1 fixes are applied: `modules.technical_view:
false` (measured -0.265 marginal Sharpe), the cash exclusion reverted
(gate census showed it dropped the BL book from 239/239 to 0/239 gate
wins), and a real risk-free rate (FRED DTB3) threaded through
`sharpe`, `utility`, `max_sharpe`, and the cash bucket's equilibrium
prior.

**Honest result: OOS Sharpe is 0.7382 -- essentially unchanged, and
`black_litterman_frozen` is numerically identical to `gmv` row for
row (same return, vol, drawdown, turnover).** This does not match
DIAGNOSTIC.md's expected range of ~1.05-1.10. A direct gate census
with the fixed pipeline (16 sampled OOS weeks) confirms the mechanism
without ambiguity: **GMV wins the utility gate in 16 out of 16 weeks,
same as before any of the three fixes.** The cash-pricing fix worked
exactly as intended on its own terms -- cash weight inside the
max_sharpe candidate book is no longer pinned to the 20% cap by a
Sharpe-ratio degeneracy (it now ranges from ~1.4% to 20% depending on
the week, tracking genuine diversification value rather than a
mathematical artifact) -- but the max_sharpe book's expected return
advantage over GMV still isn't enough to overcome its higher variance
under the utility gate's quadratic risk-aversion penalty (A=5). GMV
was the previous winner because of the cash degeneracy AND because
the gate itself structurally favors minimum variance; fixing only the
first cause left the second fully intact.

**This confirms, rather than undermines, DIAGNOSTIC.md's own Sec. 3
diagnosis** ("the gate is not a safety net on the optimizer -- it is a
second, stricter optimizer that always prefers minimum variance...
you have two conflicting objective functions in series, and the
second one wins every time") and directly motivates Tier 2 item 4
(ablate, then re-scope, the utility gate) as the necessary next step,
not an optional deepening. Tier 1 was still worth doing on its own
merits -- V3 is a measured, real negative contributor regardless of
the gate's behavior, and cash is now priced correctly rather than by
a mathematical accident -- but it was not sufficient by itself to move
the headline number, and that is reported here plainly rather than
reframed as a success.

---

**Addendum (2026-08-03, Tier 2 of DIAGNOSTIC.md's action plan: gate
recalibrated to A=2, bucket-level allocation added for comparison):**

Two Tier-2 changes are reflected in this run: (1) `black_litterman.
risk_aversion_for_utility_gate` moved from 5 to 2 in `config.yaml`,
per notebook 11's gate ablation (A=5 let GMV win the utility gate
236/236 sampled OOS weeks; A=2 let the max_sharpe book win ~84% of
the time while the gate still did real risk management the rest);
(2) `strategy.bucket_black_litterman_strategy` (DIAGNOSTIC.md action
5) is now run alongside the asset-level strategy on the identical OOS
window, added as a new `black_litterman_bucket` row.

**Honest result: the gate recalibration is a large, real improvement
on the actual frozen pipeline, not just the ablation's synthetic
harness.** `black_litterman_frozen`'s OOS Sharpe rose from Tier 1's
0.7382 (numerically identical to plain GMV) to **0.9103**
(excess-return Sharpe over the real T-bill rate: 0.6333). This is
directionally consistent with notebook 11's ablation (0.8330 ->
0.9863) though not identical in magnitude, expected since the ablation
ran on a different, shorter/differently-scoped date range and without
the anomaly override / turnover-cap interactions the full frozen
pipeline has. `black_litterman_frozen` now beats every classical
benchmark except `permanent` (gmv 0.7379, risk_parity 0.8317, hrp
0.8195, sixty_forty 0.5913, max_sharpe_static 0.6197) -- the first
time in this project's history the BL pipeline has cleared that bar.
The cost of that improvement is real: annualized vol rose from ~5.5%
(A=5, GMV-dominated) to **14.37%**, and max drawdown widened to
-16.38% (worse than `permanent`'s -12.57%), because the gate now lets
the higher-vol max_sharpe book win the large majority of weeks instead
of defaulting to minimum variance.

**The bucket-level allocation does NOT reproduce DIAGNOSTIC.md's
counterfactual, and this is reported plainly rather than reframed.**
DIAGNOSTIC.md's independent measurement (Sec 5.3) found a bucket-level
book at Sharpe 1.279 OOS, beating `permanent` (1.067) in that window.
In this fully-integrated implementation, `black_litterman_bucket`
scores **0.8602** (excess-return Sharpe 0.3847, max drawdown -13.81%)
-- close to `risk_parity` and `hrp`, but behind both
`black_litterman_frozen` (0.9103) and `permanent` (1.0841). The most
likely explanation is that DIAGNOSTIC.md's counterfactual was computed
as an isolated experiment (its own rf handling, gate behavior, and
turnover/anomaly interactions were not necessarily identical to this
repo's current full pipeline), so the two numbers are not measuring
the same object even though both are labeled "bucket-level." What is
measured here, on the real `bucket_black_litterman_strategy` wired
into the actual OOS backtest with the actual gate, real rf, and real
turnover cap, is a working but unremarkable strategy that does not
beat either `permanent` or the asset-level book in this window. Kept
alongside the asset-level path, not deleted, per `modules.
bucket_level`'s stated purpose of letting both be compared -- this is
that comparison, reported as it came out.

**Permanent still leads the whole comparison (1.0841 Sharpe), same
conclusion as every prior stage.** Tier 2's real, measured win is that
the BL pipeline's own OOS Sharpe (0.9103) now clears every classical
benchmark it previously lost to except `permanent` itself -- a
genuine improvement over Tier 1, achieved by fixing the gate's
calibration rather than any of the signal/view logic, and consistent
with DIAGNOSTIC.md Sec 3's diagnosis that the gate, not the fusion
pipeline, was the dominant bottleneck. It is not yet a result that
beats the benchmark this project is measured against.

---

**Addendum (2026-08-03, Tier 3 of DIAGNOSTIC.md's action plan: HMM
n_states via BIC, per-asset V2 hit-rate gate, capped benchmarks):**

Three more IS-only, freeze-for-OOS changes: (1) `regimes.hmm_bic_curve`
selected **n_states=4** over the candidate range {2..6} (previously a
fixed, untuned 3); (2) `meanreversion.hit_rate_eligible_tickers`
(min_hit_rate=0.55, IS-only, horizon_days=20, using the same tuned
`lookback_days=20, entry_z=2.0` the mean-reversion grid already
selected) found **8 of 22 tickers eligible for V2** (`ACWI, GLD, HYG,
IWM, QQQ, SPY, VNQ, VTI`) -- a real, larger set than an earlier ad hoc
check in this same session found (2 tickers), because that check used
different event-study parameters (`horizon_days=10`, untuned
lookback/entry_z); this notebook's number is the one actually wired
into the frozen strategy and should be treated as authoritative; (3)
`allocation.permanent`/`sixty_forty` in the benchmark comparison now
pass `cap=constraints.per_asset_cap` (0.20), so the benchmarks face
the same per-asset cap every optimizer in this pipeline already
respects.

**Honest result: essentially flat, a very small regression on the
strategy's own number, and a modestly more honest (harder) bar for
the strategy to clear.** `black_litterman_frozen` moved from 0.9103
(Tier 2) to **0.9029** (excess-return Sharpe 0.6319, both changes of
Tier 3 combined) -- within noise of Tier 2's number, not a real
improvement or regression at this sample size. The interesting result
is what happened to `permanent`: capping it (item 3) pulled its Sharpe
down from 1.0841 to **1.0571**, since `cash` (BIL alone) had been
sitting at an uncapped 25% and is now redistributed like every other
book already has to. `black_litterman_bucket` also moved down (0.8602
-> 0.8070) -- it uses the new `n_states=4` regime read (bucket-level
V1 is still active) but not the V2 gate (bucket-level has no V2 view).
`gmv`, `risk_parity`, `hrp`, `max_sharpe_static` are unchanged (none
of them touch V2, HMM n_states, or a capped `permanent`/`sixty_forty`
call); `sixty_forty`'s Sharpe is also unchanged despite now being
capped, meaning nothing in its equity/fixed_income split actually
exceeded 20% in this universe -- the cap only bound for `permanent`'s
single-ticker `cash` bucket.

**Net effect of Tier 3: a fairer, slightly harder-to-clear benchmark
(permanent's Sharpe fell ~0.027), and no measurable change to the
strategy's own performance from the HMM/V2 changes at this sample
size.** `black_litterman_frozen` (0.9029) still beats every classical
benchmark except `permanent` (1.0571) -- the same qualitative
conclusion as Tier 2, on a comparison that is now measurably fairer to
the strategy. As with every prior stage, this is reported as measured:
none of the three Tier-3 items produced the kind of large, unambiguous
move Tier 2's gate recalibration did, and that is itself informative
about where this pipeline's remaining headroom is (or isn't).

---

**Addendum (2026-08-05, ANALYSIS_V2.md Step 1: momentum as V5, IS-only
tuned and frozen):**

`momentum.momentum_view` (dual momentum: 12-1 cross-sectional rank +
absolute trend-vs-cash gate) joins V1-V3 in the fusion. IS-only grid
search (Part A3, pure vectorized rank-spread test, no model fitting)
selected `lookback_days=189, skip_days=0, top_fraction=0.5` over
2010-2021, frozen for this run.

**Honest result: this is a real, measured regression, not a wash.**
Frozen OOS Sharpe fell from Tier 3's 0.9029 to **0.7172** (excess
Sharpe 0.6319 -> 0.4357; ann. return 12.96% -> 9.57%; ann. vol
essentially unchanged, 14.7% -> 14.15%). Reported exactly as it came
out, per this project's standing rule that a measured negative result
for a course-covered signal is exactly as valuable as a positive one
(the same treatment V2 and V3 already got).

**Why, mechanically** (a 12-week sampled gate census, `momentum_view`
on vs. off, same frozen pipeline otherwise): in most sampled weeks the
utility gate still picks the same candidate book with `momentum_view`
on or off, but the max-Sharpe book's own composition differs
substantially even then -- mean L1 weight difference 0.69 across the
12 weeks, ranging 0.14 to 1.44, so V5's view is materially reshaping
`mu_BL` every week it fires, not just adding a small tilt. In 2 of the
12 sampled weeks (2022-10-14, 2023-03-03 -- both inside 2022's
rate-hike/bond-selloff stretch), the gate's choice itself flips:
`momentum_view` on selects the defensive `risk_parity` book where
`momentum_view` off still selects the max-Sharpe book. The absolute
leg (a flat negative view on every member of a bucket that fails the
trend-vs-cash gate) is the likely driver: exactly during a broad
selloff, most buckets fail that gate simultaneously, pushing mu_BL
down hard enough to tip the utility comparison toward the defensive
book at moments when, on net over this OOS window, that switch cost
more than it protected.

**This is not the same failure mode as V2 or V3.** V2 measured
~zero (a real signal too weak to move the needle); V3 measured a
clear, uniform negative contributor. V5 is neither -- it is a
believable, mechanically-explained signal that reshapes the book
every week and occasionally flips the gate's own choice, and on this
specific OOS window that reshaping cost return rather than adding it.
`modules.momentum_view` is left `true` for now, not flipped off on
the strength of this one frozen-pipeline number alone: the formal
marginal-contribution measurement belongs to the stage-11 ablation
(ANALYSIS_V2.md Step 2, next), the same venue that measured V3's
-0.265 before it was disabled -- this addendum is the frozen-pipeline
readout, not the ablation verdict.

---

**Addendum (2026-08-06, ANALYSIS_V2.md Step 2: momentum disabled per
the stage-11 ablation's formal measurement):**

`modules.momentum_view` is now `false` in `config.yaml`, set after
notebook 11's re-measured ablation (with excess Sharpe as the primary
ranking metric) found V5's marginal contribution to be **-0.2360
excess Sharpe** -- the largest negative effect ever measured in that
study, larger in relative terms than V3's original -0.265. The
IS-only momentum grid search cell above still runs and still freezes
`lookback_days=189, skip_days=0, top_fraction=0.5` into `frozen_cfg`
(kept so the code path and its tests stay exercised), but with the
module flag off, `views.momentum_views` returns an empty view set
regardless -- V5 no longer contributes to `mu_BL`.

**Honest result: frozen OOS Sharpe is 0.9491 (excess Sharpe 0.6784,
ann. return 13.74%), recovering and slightly exceeding Tier 3's
pre-momentum reading of 0.9029.** The small residual difference (not
a real methodological change -- disabling `momentum_view` reproduces
the exact pre-V5 pipeline logic) is attributable to calendar drift: a
few more trading days of live 2026 data entered the return series
between the two runs, the same effect this notebook's addenda have
repeatedly noted moves every book's numbers by a small amount, not
just the frozen strategy's. `black_litterman_frozen` again beats
every classical benchmark except `permanent` (1.0738 vs. 0.9491),
the same qualitative conclusion as Tier 2/3, now with V5 tried,
measured, found wanting on its own formal ablation terms, and
correctly turned off -- exactly the outcome V3 had in Tier 1, and
exactly the process ANALYSIS_V2.md's Step 1/2 asked for.

---

**Addendum (2026-08-06, ANALYSIS_V2.md Step 3: risk-matched
benchmark):**

Levering `permanent` (financed at the prevailing T-bill rate) up to
ATLAS's own realized volatility requires **1.667x** leverage. As
expected, leverage financed at `rf` is (essentially) Sharpe-invariant
in practice, not just in theory: levered `permanent`'s excess Sharpe
(0.6225) is identical to unlevered `permanent`'s (0.6225) to four
decimal places over this real, discretely-compounded 4.5-year series.

**Honest result: ATLAS now shows real, measured alpha over the
risk-matched benchmark -- +0.93% per year (13.74% vs. 12.81% annual
return at matched 14.7% volatility).** This is a materially different
finding from ANALYSIS_V2.md Sec 2's original ~+0.02%/yr (statistically
indistinguishable from zero), because that number was computed before
Tier 3's gate/HMM/V2 refinements and before V5 was tried and
correctly rejected -- this notebook's own frozen pipeline has moved
since. ATLAS's excess Sharpe (0.6784) also now clears the risk-matched
benchmark's (0.6225) by a real margin, not the ~0.004 gap ANALYSIS_V2.md
found. A second, independent point in ATLAS's favor: at the SAME
volatility, ATLAS's max drawdown (-16.68%) is meaningfully shallower
than the risk-matched levered permanent's (-22.08%) -- the extra risk
ATLAS takes is evidently better-timed (regime-aware, gated), not just
scaled up uniformly the way pure leverage is.

**This does not overturn the project's honest headline** (`permanent`
unlevered still posts the higher total Sharpe, 1.07 vs. 0.95, because
Sharpe doesn't care how much risk you take and `permanent` takes
much less) -- but it does answer the harder, fairer question
ANALYSIS_V2.md Sec 2 posed: once risk is matched, does ATLAS's extra
return reflect genuine skill or just leverage? On the current,
fully-measured pipeline (V3 off, gate at A=2, V2 restricted, HMM
BIC-selected, V5 tried and rejected), the answer is now a real,
positive, if modest, yes.

## Deliverable exports (S11 Step 5, ANALYSIS_V2.md action 5)

Saves the canonical charts (now built via `plotting.py` rather than hand-rolled per cell, see above) to `reports/figures/` and a compact, machine-readable snapshot of this run's headline numbers and frozen parameters to `reports/results/`.

In [ ]:
import json

FIGURES_DIR = data.PROJECT_ROOT / "reports" / "figures"
RESULTS_DIR = data.PROJECT_ROOT / "reports" / "results"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

fig1 = plotting.equity_curves(
    oos_returns, oos_start=oos_start,
    highlight="black_litterman_frozen",
    title="OOS equity curves: frozen Black-Litterman vs. benchmarks",
)
fig1.savefig(
    FIGURES_DIR / "fig01_equity_curves.png", dpi=150, bbox_inches="tight"
)

fig2 = plotting.drawdown_curves(
    oos_returns, oos_start=oos_start,
    highlight="black_litterman_frozen",
    title="OOS drawdowns: frozen Black-Litterman vs. benchmarks",
)
fig2.savefig(
    FIGURES_DIR / "fig02_drawdowns.png", dpi=150, bbox_inches="tight"
)

fig3 = plotting.weight_evolution(
    frozen_result.weights.loc[oos_start:],
    title="Frozen Black-Litterman: OOS weight evolution",
)
fig3.savefig(
    FIGURES_DIR / "fig03_weight_evolution.png", dpi=150, bbox_inches="tight"
)

risk_matched_returns = {
    "permanent": permanent_oos,
    levered_label: levered_permanent_returns,
    "black_litterman_frozen (ATLAS)": atlas_oos,
}
fig4 = plotting.equity_curves(
    risk_matched_returns,
    highlight="black_litterman_frozen (ATLAS)",
    title="Risk-matched comparison: ATLAS vs. permanent vs. levered permanent",
)
fig4.savefig(
    FIGURES_DIR / "fig04_risk_matched_comparison.png",
    dpi=150, bbox_inches="tight",
)

print(f"Saved 4 figures to {FIGURES_DIR}")

In [ ]:
results_summary = {
    "generated": pd.Timestamp.now().isoformat(),
    "oos_window": {
        "start": str(oos_start.date()),
        "end": str(backtest_returns.index.max().date()),
    },
    "comparison_table": comparison_table.round(4).to_dict(
        orient="index"
    ),
    "risk_matched": {
        "leverage": round(float(leverage), 4),
        "alpha_annual": round(float(alpha), 6),
        "table": risk_matched_table.round(4).to_dict(orient="index"),
    },
    "frozen_params": {
        "hmm_n_states": frozen_cfg["regimes"]["hmm"]["n_states"],
        "meanreversion": {
            "lookback_days": frozen_cfg["meanreversion"]["lookback_days"],
            "entry_z": frozen_cfg["meanreversion"]["entry_z"],
        },
        "momentum": frozen_cfg["momentum"],
        "modules": frozen_cfg["modules"],
        "gate": frozen_cfg["black_litterman"]["gate"],
        "risk_aversion_for_utility_gate": frozen_cfg["black_litterman"][
            "risk_aversion_for_utility_gate"
        ],
    },
}
with open(RESULTS_DIR / "results_summary.json", "w") as f:
    json.dump(results_summary, f, indent=2, default=str)
print(f"Saved results_summary.json to {RESULTS_DIR}")

---

**Addendum (2026-08-07, ANALYSIS_V2.md Step 5: plotting.py, exported
deliverables):**

The equity-curve, drawdown, and weight-evolution charts above now
call `plotting.equity_curves`/`drawdown_curves`/`weight_evolution`
instead of hand-rolled matplotlib per cell (12 tests in
`test_plotting.py`). The four canonical figures are exported to
`reports/figures/` (`fig01_equity_curves.png` through
`fig04_risk_matched_comparison.png`, ~150-270KB each, visually
confirmed) and a machine-readable `results_summary.json` (comparison
table, risk-matched alpha/leverage, every frozen parameter) to
`reports/results/` -- both were empty placeholders before this run.

This is a routine re-execution after Step 4's config fixes (none of
which change strategy behavior) and the plotting/export wiring itself
(also behavior-neutral for the strategy) -- the numbers moved only by
the usual small amount from a few more days of live 2026 data entering
the series: frozen Sharpe 0.9491 -> 0.9439, excess Sharpe 0.6784 ->
0.6732, risk-matched alpha +0.93%/yr -> +0.80%/yr. All within the
range of run-to-run calendar drift this notebook has repeatedly
documented, not a new finding.

**Scope note, disclosed honestly:** `plotting.py` was wired into
notebook 09 only, not notebooks 02-08/10/11 (ANALYSIS_V2.md's
"hand-rolled across notebooks 02-11" framing). Notebook 11's own
equity-curve and marginal-contribution bar-chart cells still hand-roll
the same patterns `plotting.py` now covers -- a natural, low-risk
follow-up (swap cell bodies for `plotting.equity_curves`/
`marginal_contribution_bars` calls, no re-derivation needed) not done
here given this stage's time budget. `strategy_legacy.py` (stage 3/5/6
wrappers) and its test split are complete and green.